In [2]:
from multiagent_factcheck import ExpertConfig, DecisionConfig, run_multiagent_on_csv

experts = [
    ExpertConfig(
        name="Politics&Elections SME",
        model_name="meta-llama/Llama-3.1-8B-Instruct",
        weight=1.4,
        max_new_tokens=112,
        top_p=1.0,
        prompt_template="""You are a subject-matter expert in U.S. politics, elections, legislation, and public records.

Task: Verify the claim with objective, publicly verifiable facts (official roll calls, bill texts, FEC/CRP, governor/congress records, reputable outlets). If evidence is insufficient/ambiguous, output "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Focus domains (if present): elections, campaign-finance, voting records, state/federal legislation, party platforms.

Input:
Claim: {statement}
{subject_block}
JSON:"""
    ),
    ExpertConfig(
        name="Economy&Taxes SME",
        model_name="meta-llama/Llama-3.1-8B-Instruct",
        weight=1.3,
        max_new_tokens=112,
        top_p=1.0,
        prompt_template="""You are a subject-matter expert in macroeconomics, labor stats, budgets, and taxation.

Task: Verify using BLS, BEA, CBO, JCT, Treasury/IRS, OMB, WTO/IMF/World Bank, and reputable reports. Quantify time ranges and definitions. If data is unclear or cherry-picked, output "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Focus domains: taxes, jobs, economy, trade, social security/medicare finances, government budget.

Input:
Claim: {statement}
{subject_block}
JSON:"""
    ),
    ExpertConfig(
        name="Health&Science SME",
        model_name="meta-llama/Llama-3.1-8B-Instruct",
        weight=1.3,
        max_new_tokens=112,
        top_p=1.0,
        prompt_template="""You are a subject-matter expert for healthcare policy, public health, and scientific claims.

Task: Verify with CDC, FDA, NIH, CMS, WHO, Cochrane, PubMed-indexed studies, and official rulemaking. Distinguish correlation vs. causation and specify years/locations. If evidence is weak, output "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Focus domains: health-care, ACA/Medicare/Medicaid, vaccines, drugs/overdoses, abortion/stem cells, general science.

Input:
Claim: {statement}
{subject_block}
JSON:"""
    ),
]

decision = DecisionConfig(
    model_name="meta-llama/Llama-3.1-8B-Instruct",
    prompt_template="""You are a senior fact-checking adjudicator.
You receive multiple expert analyses, each with a verdict, explanation, and weight.

Task:
- Evaluate the experts' reasoning quality, factual grounding, and agreement with publicly verifiable evidence.
- Consider the explicit weights assigned to each expert.
- Prioritize well-supported, domain-relevant arguments over unsupported or vague statements.
- If the majority view (weighted) is inconclusive or poorly justified, choose "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Input JSON (array of expert objects):
{experts_json}
JSON:"""
)


df_out, metrics = run_multiagent_on_csv(
    csv_path="own_datasets/test_binary_labels.csv",
    experts=experts,
    decision=decision,
    statement_col="statement",
    subject_col="subjects",
    label_col="label",
    limit= 500,
    save_path="results/multiagent_results.csv",
    return_intermediates=True
)

df_out.head()

2025-08-10 06:16:32.402910: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754799392.422692  269807 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1754799392.428041  269807 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1754799392.442019  269807 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1754799392.442046  269807 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1754799392.442048  269807 computation_placer.cc:177] computation placer alr

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🔎 Fact-checking: 100%|██████████| 500/500 [2:41:33<00:00, 19.39s/it]  


📊 Evaluation
accuracy: 0.466
precision_true: 0.560
recall_true: 0.183
f1_true: 0.276
confusion_matrix: [[ 51 227]
 [ 40 182]]

Classification report:
               precision    recall  f1-score   support

        True       0.56      0.18      0.28       278
       False       0.44      0.82      0.58       222

    accuracy                           0.47       500
   macro avg       0.50      0.50      0.43       500
weighted avg       0.51      0.47      0.41       500

💾 Ergebnisse gespeichert unter: results/multiagent_results.csv


,label,statement,subjects,prediction,explanation,experts
0,True,Building a wall on the U.S . Mexico border wil...,immigration,False,2-4 explaining sentences,"[{'name': 'Politics&Elections SME', 'weight': ..."
1,False,Wisconsin is on pace to double the number of l...,jobs,False,"Since all experts lack explanation, the majori...","[{'name': 'Politics&Elections SME', 'weight': ..."
2,False,Says John McCain has done nothing to help the ...,"military,veterans,voting-record",False,2-4 explaining sentences,"[{'name': 'Politics&Elections SME', 'weight': ..."
3,True,Suzanne Bonamici supports a plan that will cut...,"medicare,message-machine-2012,campaign-adverti...",False,2-4 explaining sentences,"[{'name': 'Politics&Elections SME', 'weight': ..."
4,False,When asked by a reporter whether hes at the ce...,"campaign-finance,legal-issues,campaign-adverti...",True,Both Politics&Elections SME and Economy&Taxes ...,"[{'name': 'Politics&Elections SME', 'weight': ..."


In [4]:
from multiagent_factcheck import ExpertConfig, DecisionConfig, run_multiagent_on_csv

experts = [
    ExpertConfig(
        name="Politics&Elections SME",
        model_name="deepseek-ai/deepseek-llm-7b-base",
        weight=1.4,
        max_new_tokens=112,
        top_p=1.0,
        prompt_template="""You are a subject-matter expert in U.S. politics, elections, legislation, and public records.

Task: Verify the claim with objective, publicly verifiable facts (official roll calls, bill texts, FEC/CRP, governor/congress records, reputable outlets). If evidence is insufficient/ambiguous, output "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Focus domains (if present): elections, campaign-finance, voting records, state/federal legislation, party platforms.

Input:
Claim: {statement}
{subject_block}
JSON:"""
    ),
    ExpertConfig(
        name="Economy&Taxes SME",
        model_name="deepseek-ai/deepseek-llm-7b-base",
        weight=1.3,
        max_new_tokens=112,
        top_p=1.0,
        prompt_template="""You are a subject-matter expert in macroeconomics, labor stats, budgets, and taxation.

Task: Verify using BLS, BEA, CBO, JCT, Treasury/IRS, OMB, WTO/IMF/World Bank, and reputable reports. Quantify time ranges and definitions. If data is unclear or cherry-picked, output "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Focus domains: taxes, jobs, economy, trade, social security/medicare finances, government budget.

Input:
Claim: {statement}
{subject_block}
JSON:"""
    ),
    ExpertConfig(
        name="Health&Science SME",
        model_name="deepseek-ai/deepseek-llm-7b-base",
        weight=1.3,
        max_new_tokens=112,
        top_p=1.0,
        prompt_template="""You are a subject-matter expert for healthcare policy, public health, and scientific claims.

Task: Verify with CDC, FDA, NIH, CMS, WHO, Cochrane, PubMed-indexed studies, and official rulemaking. Distinguish correlation vs. causation and specify years/locations. If evidence is weak, output "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Focus domains: health-care, ACA/Medicare/Medicaid, vaccines, drugs/overdoses, abortion/stem cells, general science.

Input:
Claim: {statement}
{subject_block}
JSON:"""
    ),
]

decision = DecisionConfig(
    model_name="deepseek-ai/deepseek-llm-7b-base",
    prompt_template="""You are a senior fact-checking adjudicator.
You receive multiple expert analyses, each with a verdict, explanation, and weight.

Task:
- Evaluate the experts' reasoning quality, factual grounding, and agreement with publicly verifiable evidence.
- Consider the explicit weights assigned to each expert.
- Prioritize well-supported, domain-relevant arguments over unsupported or vague statements.
- If the majority view (weighted) is inconclusive or poorly justified, choose "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Input JSON (array of expert objects):
{experts_json}
JSON:"""
)


df_out, metrics = run_multiagent_on_csv(
    csv_path="../own_datasets/test_binary_labels.csv",
    experts=experts,
    decision=decision,
    statement_col="statement",
    subject_col="subjects",
    label_col="label",
    limit= 5,
    save_path="results/multiagent_results_gemma.csv",
    return_intermediates=True
)

df_out.head()

🔎 Fact-checking:   0%|          | 0/5 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

🔎 Fact-checking: 100%|██████████| 5/5 [01:21<00:00, 16.39s/it]


📊 Evaluation
accuracy: 0.600
precision_true: 0.000
recall_true: 0.000
f1_true: 0.000
confusion_matrix: [[0 2]
 [0 3]]

Classification report:
               precision    recall  f1-score   support

        True       0.00      0.00      0.00         2
       False       0.60      1.00      0.75         3

    accuracy                           0.60         5
   macro avg       0.30      0.50      0.38         5
weighted avg       0.36      0.60      0.45         5

💾 Ergebnisse gespeichert unter: results/multiagent_results_gemma.csv



/opt/python/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/python/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/python/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/python/lib/python3.12/site-packages/

,label,statement,subjects,prediction,explanation,experts
0,True,Building a wall on the U.S . Mexico border wil...,immigration,False,No explanation provided.,"[{'name': 'Politics&Elections SME', 'weight': ..."
1,False,Wisconsin is on pace to double the number of l...,jobs,False,No explanation provided.,"[{'name': 'Politics&Elections SME', 'weight': ..."
2,False,Says John McCain has done nothing to help the ...,"military,veterans,voting-record",False,No explanation provided.,"[{'name': 'Politics&Elections SME', 'weight': ..."
3,True,Suzanne Bonamici supports a plan that will cut...,"medicare,message-machine-2012,campaign-adverti...",False,"{ ""name"": ""Politics&Elections SME"", ""weight"": ...","[{'name': 'Politics&Elections SME', 'weight': ..."
4,False,When asked by a reporter whether hes at the ce...,"campaign-finance,legal-issues,campaign-adverti...",False,No explanation provided.,"[{'name': 'Politics&Elections SME', 'weight': ..."


In [4]:
from multiagent_factcheck_2 import ExpertConfig, DecisionConfig, run_multiagent_on_csv

experts = [
    ExpertConfig(
        name="Politics&Elections SME",
        model_name="meta-llama/Llama-3.1-8B-Instruct",
        weight=1.4,
        max_new_tokens=112,
        top_p=1.0,
        device_map="cuda:0",
        prompt_template="""You are a subject-matter expert in U.S. politics, elections, legislation, and public records.

Task: Verify the claim with objective, publicly verifiable facts (official roll calls, bill texts, FEC/CRP, governor/congress records, reputable outlets). If evidence is insufficient/ambiguous, output "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Focus domains (if present): elections, campaign-finance, voting records, state/federal legislation, party platforms.

Input:
Claim: {statement}
{subject_block}
JSON:"""
    ),
    ExpertConfig(
        name="Economy&Taxes SME",
        model_name="meta-llama/Llama-3.1-8B-Instruct",
        weight=1.3,
        max_new_tokens=112,
        top_p=1.0,
        device_map="cuda:0",
        prompt_template="""You are a subject-matter expert in macroeconomics, labor stats, budgets, and taxation.

Task: Verify using BLS, BEA, CBO, JCT, Treasury/IRS, OMB, WTO/IMF/World Bank, and reputable reports. Quantify time ranges and definitions. If data is unclear or cherry-picked, output "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Focus domains: taxes, jobs, economy, trade, social security/medicare finances, government budget.

Input:
Claim: {statement}
{subject_block}
JSON:"""
    ),
    ExpertConfig(
        name="Health&Science SME",
        model_name="meta-llama/Llama-3.1-8B-Instruct",
        weight=1.3,
        max_new_tokens=112,
        top_p=1.0,
        device_map="cuda:0",
        prompt_template="""You are a subject-matter expert for healthcare policy, public health, and scientific claims.

Task: Verify with CDC, FDA, NIH, CMS, WHO, Cochrane, PubMed-indexed studies, and official rulemaking. Distinguish correlation vs. causation and specify years/locations. If evidence is weak, output "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Focus domains: health-care, ACA/Medicare/Medicaid, vaccines, drugs/overdoses, abortion/stem cells, general science.

Input:
Claim: {statement}
{subject_block}
JSON:"""
    ),
]

decision = DecisionConfig(model_name="meta-llama/Llama-3.1-8B-Instruct")

df_out, metrics = run_multiagent_on_csv(
    csv_path="own_datasets/test_binary_labels.csv",
    experts=experts,
    decision=decision,
    statement_col="statement",
    subject_col="subjects",
    label_col="label",
    limit= 500,
    save_path="results/multiagent_results_2.csv",
    return_intermediates=True
)

df_out.head()

🔎 Fact-checking:   0%|          | 0/500 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🔎 Fact-checking: 100%|██████████| 500/500 [2:42:19<00:00, 19.48s/it]  


📊 Evaluation
accuracy: 0.466
precision_true: 0.824
recall_true: 0.050
f1_true: 0.095
confusion_matrix: [[ 14 264]
 [  3 219]]

Classification report:
               precision    recall  f1-score   support

        True       0.82      0.05      0.09       278
       False       0.45      0.99      0.62       222

    accuracy                           0.47       500
   macro avg       0.64      0.52      0.36       500
weighted avg       0.66      0.47      0.33       500

💾 Ergebnisse gespeichert unter: results/multiagent_results.csv


,label,statement,subjects,prediction,explanation,experts
0,True,Building a wall on the U.S . Mexico border wil...,immigration,False,The majority of experts agree that building a ...,"[{'name': 'Politics&Elections SME', 'weight': ..."
1,False,Wisconsin is on pace to double the number of l...,jobs,False,The majority of experts agree that the claim i...,"[{'name': 'Politics&Elections SME', 'weight': ..."
2,False,Says John McCain has done nothing to help the ...,"military,veterans,voting-record",False,All experts agree that John McCain has done no...,"[{'name': 'Politics&Elections SME', 'weight': ..."
3,True,Suzanne Bonamici supports a plan that will cut...,"medicare,message-machine-2012,campaign-adverti...",False,The majority of experts agree that Suzanne Bon...,"[{'name': 'Politics&Elections SME', 'weight': ..."
4,False,When asked by a reporter whether hes at the ce...,"campaign-finance,legal-issues,campaign-adverti...",False,The final verdict is False because the majorit...,"[{'name': 'Politics&Elections SME', 'weight': ..."


In [ ]:
from multiagent_factcheck_2 import ExpertConfig, DecisionConfig, run_multiagent_on_csv

experts = [
    ExpertConfig(
        name="Politics&Elections SME",
        model_name="models/finetuned_with_subjects_json_style_politics_expert",
        weight=1.4,
        max_new_tokens=112,
        top_p=1.0,
        #device_map="cuda:0",
        prompt_template="""You are a subject-matter expert in U.S. politics, elections, legislation, and public records.

Task: Verify the claim with objective, publicly verifiable facts (official roll calls, bill texts, FEC/CRP, governor/congress records, reputable outlets). If evidence is insufficient/ambiguous, output "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Focus domains (if present): elections, campaign-finance, voting records, state/federal legislation, party platforms.

Input:
Claim: {statement}
{subject_block}
JSON:"""
    ),
    ExpertConfig(
        name="Economy&Taxes SME",
        model_name="models/finetuned_with_subjects_json_style_economy_expert",
        weight=1.3,
        max_new_tokens=112,
        top_p=1.0,
        #device_map="cuda:0",
        prompt_template="""You are a subject-matter expert in macroeconomics, labor stats, budgets, and taxation.

Task: Verify using BLS, BEA, CBO, JCT, Treasury/IRS, OMB, WTO/IMF/World Bank, and reputable reports. Quantify time ranges and definitions. If data is unclear or cherry-picked, output "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Focus domains: taxes, jobs, economy, trade, social security/medicare finances, government budget.

Input:
Claim: {statement}
{subject_block}
JSON:"""
    ),
    ExpertConfig(
        name="Health&Science SME",
        model_name="models/finetuned_with_subjects_json_style_health_expert",
        weight=1.3,
        max_new_tokens=112,
        top_p=1.0,
        #device_map="cuda:0",
        prompt_template="""You are a subject-matter expert for healthcare policy, public health, and scientific claims.

Task: Verify with CDC, FDA, NIH, CMS, WHO, Cochrane, PubMed-indexed studies, and official rulemaking. Distinguish correlation vs. causation and specify years/locations. If evidence is weak, output "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Focus domains: health-care, ACA/Medicare/Medicaid, vaccines, drugs/overdoses, abortion/stem cells, general science.

Input:
Claim: {statement}
{subject_block}
JSON:"""
    ),
]

decision = DecisionConfig(model_name="meta-llama/Llama-3.1-8B-Instruct")

df_out, metrics = run_multiagent_on_csv(
    csv_path="own_datasets/test_binary_labels.csv",
    experts=experts,
    decision=decision,
    statement_col="statement",
    subject_col="subjects",
    label_col="label",
    limit= 500,
    save_path="results/multiagent_results_finetuned.csv",
    return_intermediates=True
)

df_out.head()

2025-08-16 11:23:53.983537: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1755336234.003485  300946 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1755336234.008214  300946 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1755336234.020668  300946 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1755336234.020680  300946 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1755336234.020682  300946 computation_placer.cc:177] computation placer alr

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🔎 Fact-checking:   0%|          | 2/500 [01:31<5:40:24, 41.01s/it]

In [2]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

# Pfad zur Ergebnis-CSV
csv_path = "results/multiagent_results_finetuned.csv"

# CSV einlesen
df = pd.read_csv(csv_path)

# Ground Truth und Vorhersagen als Strings normalisieren
y_true = df["label"].astype(str)
y_pred = df["prediction"].astype(str)

# Metriken berechnen
metrics = {
    "accuracy": accuracy_score(y_true, y_pred),
    "precision_true": precision_score(y_true, y_pred, pos_label="True"),
    "recall_true": recall_score(y_true, y_pred, pos_label="True"),
    "f1_true": f1_score(y_true, y_pred, pos_label="True"),
    "report": classification_report(y_true, y_pred, labels=["True","False"], target_names=["True","False"]),
    "confusion_matrix": confusion_matrix(y_true, y_pred, labels=["True", "False"])
}

# Ergebnisse ausgeben
print("\n📊 Evaluation")
for k, v in metrics.items():
    if k != "report":
        print(f"{k}: {v:.3f}" if isinstance(v, float) else f"{k}: {v}")
print("\nClassification report:\n", metrics["report"])



📊 Evaluation
accuracy: 0.526
precision_true: 0.634
recall_true: 0.349
f1_true: 0.450
confusion_matrix: [[ 97 181]
 [ 56 166]]

Classification report:
               precision    recall  f1-score   support

        True       0.63      0.35      0.45       278
       False       0.48      0.75      0.58       222

    accuracy                           0.53       500
   macro avg       0.56      0.55      0.52       500
weighted avg       0.56      0.53      0.51       500



In [1]:
from multiagent_factcheck_2 import ExpertConfig, DecisionConfig, run_multiagent_on_csv

experts = [
    ExpertConfig(
        name="Politics&Elections SME",
        model_name="models/finetuned_json_style_politics_expert_short_prompt",
        weight=1.4,
        max_new_tokens=112,
        top_p=1.0,
        #device_map="cuda:0",
        prompt_template="""You are a subject-matter expert in U.S. politics, elections, legislation, and public records.

Task: Verify the claim with objective, publicly verifiable facts (official roll calls, bill texts, FEC/CRP, governor/congress records, reputable outlets). If evidence is insufficient/ambiguous, output "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Focus domains (if present): elections, campaign-finance, voting records, state/federal legislation, party platforms.

Input:
Claim: {statement}
{subject_block}
JSON:"""
    ),
    ExpertConfig(
        name="Economy&Taxes SME",
        model_name="models/finetuned_json_style_economy_expert_short_prompt",
        weight=1.3,
        max_new_tokens=112,
        top_p=1.0,
        #device_map="cuda:0",
        prompt_template="""You are a subject-matter expert in macroeconomics, labor stats, budgets, and taxation.

Task: Verify using BLS, BEA, CBO, JCT, Treasury/IRS, OMB, WTO/IMF/World Bank, and reputable reports. Quantify time ranges and definitions. If data is unclear or cherry-picked, output "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Focus domains: taxes, jobs, economy, trade, social security/medicare finances, government budget.

Input:
Claim: {statement}
{subject_block}
JSON:"""
    ),
    ExpertConfig(
        name="Health&Science SME",
        model_name="models/finetuned_json_style_health_expert_short_prompt",
        weight=1.3,
        max_new_tokens=112,
        top_p=1.0,
        #device_map="cuda:0",
        prompt_template="""You are a subject-matter expert for healthcare policy, public health, and scientific claims.

Task: Verify with CDC, FDA, NIH, CMS, WHO, Cochrane, PubMed-indexed studies, and official rulemaking. Distinguish correlation vs. causation and specify years/locations. If evidence is weak, output "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Focus domains: health-care, ACA/Medicare/Medicaid, vaccines, drugs/overdoses, abortion/stem cells, general science.

Input:
Claim: {statement}
{subject_block}
JSON:"""
    ),
]

decision = DecisionConfig(model_name="meta-llama/Llama-3.1-8B-Instruct")

df_out, metrics = run_multiagent_on_csv(
    csv_path="own_datasets/test_binary_labels.csv",
    experts=experts,
    decision=decision,
    statement_col="statement",
    subject_col="subjects",
    label_col="label",
    limit= 500,
    save_path="results/multiagent_results_finetuned_short_prompt.csv",
    return_intermediates=True
)

df_out.head()

2025-08-17 08:39:29.971821: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1755412769.995634  308065 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1755412770.000419  308065 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1755412770.013799  308065 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1755412770.013814  308065 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1755412770.013816  308065 computation_placer.cc:177] computation placer alr

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🔎 Fact-checking: 100%|██████████| 500/500 [3:29:49<00:00, 25.18s/it]  


📊 Evaluation
accuracy: 0.492
precision_true: 0.731
recall_true: 0.137
f1_true: 0.230
confusion_matrix: [[ 38 240]
 [ 14 208]]

Classification report:
               precision    recall  f1-score   support

        True       0.73      0.14      0.23       278
       False       0.46      0.94      0.62       222

    accuracy                           0.49       500
   macro avg       0.60      0.54      0.43       500
weighted avg       0.61      0.49      0.40       500

💾 Ergebnisse gespeichert unter: results/multiagent_results_finetuned_short_prompt.csv


,label,statement,subjects,prediction,explanation,experts
0,True,Building a wall on the U.S . Mexico border wil...,immigration,False,Politics&Elections SME has the highest weight ...,"[{'name': 'Politics&Elections SME', 'weight': ..."
1,False,Wisconsin is on pace to double the number of l...,jobs,False,"{""verdict"": ""False"", ""explanation"": ""The major...","[{'name': 'Politics&Elections SME', 'weight': ..."
2,False,Says John McCain has done nothing to help the ...,"military,veterans,voting-record",True,The verdict is True because the Economy&Taxes ...,"[{'name': 'Politics&Elections SME', 'weight': ..."
3,True,Suzanne Bonamici supports a plan that will cut...,"medicare,message-machine-2012,campaign-adverti...",False,Politics&Elections SME's verdict is more convi...,"[{'name': 'Politics&Elections SME', 'weight': ..."
4,False,When asked by a reporter whether hes at the ce...,"campaign-finance,legal-issues,campaign-adverti...",False,```python import json def aggregate_expert_vot...,"[{'name': 'Politics&Elections SME', 'weight': ..."


In [1]:
from multiagent_factcheck_3 import ExpertConfig, DecisionConfig, run_multiagent_on_csv

experts = [
    ExpertConfig(
        name="Politics&Elections SME",
        model_name="meta-llama/Llama-3.1-8B-Instruct",
        weight=1.4,
        max_new_tokens=112,
        top_p=1.0,
        prompt_template="""You are a subject-matter expert in U.S. politics, elections, legislation, and public records.

Task: Verify the claim with objective, publicly verifiable facts (official roll calls, bill texts, FEC/CRP, governor/congress records, reputable outlets). If evidence is insufficient/ambiguous, output "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Focus domains (if present): elections, campaign-finance, voting records, state/federal legislation, party platforms.

Input:
Claim: {statement}
{subject_block}
JSON:"""
    ),
    ExpertConfig(
        name="Economy&Taxes SME",
        model_name="meta-llama/Llama-3.1-8B-Instruct",
        weight=1.3,
        max_new_tokens=112,
        top_p=1.0,
        prompt_template="""You are a subject-matter expert in macroeconomics, labor stats, budgets, and taxation.

Task: Verify using BLS, BEA, CBO, JCT, Treasury/IRS, OMB, WTO/IMF/World Bank, and reputable reports. Quantify time ranges and definitions. If data is unclear or cherry-picked, output "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Focus domains: taxes, jobs, economy, trade, social security/medicare finances, government budget.

Input:
Claim: {statement}
{subject_block}
JSON:"""
    ),
    ExpertConfig(
        name="Health&Science SME",
        model_name="meta-llama/Llama-3.1-8B-Instruct",
        weight=1.3,
        max_new_tokens=112,
        top_p=1.0,
        prompt_template="""You are a subject-matter expert for healthcare policy, public health, and scientific claims.

Task: Verify with CDC, FDA, NIH, CMS, WHO, Cochrane, PubMed-indexed studies, and official rulemaking. Distinguish correlation vs. causation and specify years/locations. If evidence is weak, output "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Focus domains: health-care, ACA/Medicare/Medicaid, vaccines, drugs/overdoses, abortion/stem cells, general science.

Input:
Claim: {statement}
{subject_block}
JSON:"""
    ),
]

decision = DecisionConfig(
    model_name="meta-llama/Llama-3.1-8B-Instruct",
    prompt_template="""You are a senior fact-checking adjudicator.
You receive multiple expert analyses, each with a verdict, explanation, and weight.

Task:
- Evaluate the experts' reasoning quality, factual grounding, and agreement with publicly verifiable evidence.
- Consider the explicit weights assigned to each expert.
- Prioritize well-supported, domain-relevant arguments over unsupported or vague statements.
- If the majority view (weighted) is inconclusive or poorly justified, choose "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Input JSON (array of expert objects):
{experts_json}
JSON:"""
)


df_out, metrics = run_multiagent_on_csv(
    csv_path="own_datasets/test_binary_labels.csv",
    experts=experts,
    decision=decision,
    statement_col="statement",
    subject_col="subjects",
    label_col="label",
    limit= 500,
    save_path="results/multiagent_results_3.csv",
    return_intermediates=True
)

df_out.head()

2025-08-24 06:43:27.438816: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756010607.458029  312421 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756010607.463874  312421 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1756010607.479754  312421 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1756010607.479769  312421 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1756010607.479771  312421 computation_placer.cc:177] computation placer alr

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🔎 Fact-checking: 100%|██████████| 500/500 [3:54:07<00:00, 28.10s/it]  


📊 Evaluation
accuracy: 0.506
precision_true: 0.564
recall_true: 0.493
f1_true: 0.526
confusion_matrix: [[137 141]
 [106 116]]

Classification report:
               precision    recall  f1-score   support

        True       0.56      0.49      0.53       278
       False       0.45      0.52      0.48       222

    accuracy                           0.51       500
   macro avg       0.51      0.51      0.51       500
weighted avg       0.51      0.51      0.51       500

💾 Ergebnisse gespeichert unter: results/multiagent_results_3.csv


,label,statement,subjects,prediction,explanation,experts
0,True,Building a wall on the U.S . Mexico border wil...,immigration,True,The majority of the experts agree that the U.S...,"[{'name': 'Politics&Elections SME', 'weight': ..."
1,False,Wisconsin is on pace to double the number of l...,jobs,False,"{""experts"": [ { ""name"": ""Politics&Elections SM...","[{'name': 'Politics&Elections SME', 'weight': ..."
2,False,Says John McCain has done nothing to help the ...,"military,veterans,voting-record",True,The majority of experts agree that John McCain...,"[{'name': 'Politics&Elections SME', 'weight': ..."
3,True,Suzanne Bonamici supports a plan that will cut...,"medicare,message-machine-2012,campaign-adverti...",False,"{""experts"": [ { ""name"": ""Politics&Elections SM...","[{'name': 'Politics&Elections SME', 'weight': ..."
4,False,When asked by a reporter whether hes at the ce...,"campaign-finance,legal-issues,campaign-adverti...",True,The majority of experts agree that Gov. Scott ...,"[{'name': 'Politics&Elections SME', 'weight': ..."


In [1]:
from multiagent_factcheck_4 import ExpertConfig, DecisionConfig, run_multiagent_on_csv

experts = [
    ExpertConfig(
        name="Politics&Elections SME",
        model_name="meta-llama/Llama-3.1-8B-Instruct",
        weight=1.4,
        max_new_tokens=112,
        top_p=1.0,
        prompt_template="""You are a subject-matter expert in U.S. politics, elections, legislation, and public records.

Task: Verify the claim with objective, publicly verifiable facts (official roll calls, bill texts, FEC/CRP, governor/congress records, reputable outlets). If evidence is insufficient/ambiguous, output "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Focus domains (if present): elections, campaign-finance, voting records, state/federal legislation, party platforms.

Input:
Claim: {statement}
{subject_block}
JSON:"""
    ),
    ExpertConfig(
        name="Economy&Taxes SME",
        model_name="meta-llama/Llama-3.1-8B-Instruct",
        weight=1.3,
        max_new_tokens=112,
        top_p=1.0,
        prompt_template="""You are a subject-matter expert in macroeconomics, labor stats, budgets, and taxation.

Task: Verify using BLS, BEA, CBO, JCT, Treasury/IRS, OMB, WTO/IMF/World Bank, and reputable reports. Quantify time ranges and definitions. If data is unclear or cherry-picked, output "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Focus domains: taxes, jobs, economy, trade, social security/medicare finances, government budget.

Input:
Claim: {statement}
{subject_block}
JSON:"""
    ),
    ExpertConfig(
        name="Health&Science SME",
        model_name="meta-llama/Llama-3.1-8B-Instruct",
        weight=1.3,
        max_new_tokens=112,
        top_p=1.0,
        prompt_template="""You are a subject-matter expert for healthcare policy, public health, and scientific claims.

Task: Verify with CDC, FDA, NIH, CMS, WHO, Cochrane, PubMed-indexed studies, and official rulemaking. Distinguish correlation vs. causation and specify years/locations. If evidence is weak, output "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Focus domains: health-care, ACA/Medicare/Medicaid, vaccines, drugs/overdoses, abortion/stem cells, general science.

Input:
Claim: {statement}
{subject_block}
JSON:"""
    ),
]

decision = DecisionConfig(
    model_name="meta-llama/Llama-3.1-8B-Instruct",
    prompt_template="""You are a senior fact-checking adjudicator.
You receive multiple expert analyses, each with a verdict, explanation, and weight.

Task:
- Evaluate the experts' reasoning quality, factual grounding, and agreement with publicly verifiable evidence.
- Consider the explicit weights assigned to each expert.
- Prioritize well-supported, domain-relevant arguments over unsupported or vague statements.
- If the majority view (weighted) is inconclusive or poorly justified, choose "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Input JSON (array of expert objects):
{experts_json}
JSON:"""
)


df_out, metrics = run_multiagent_on_csv(
    csv_path="own_datasets/test_binary_labels.csv",
    experts=experts,
    decision=decision,
    statement_col="statement",
    subject_col="subjects",
    label_col="label",
    limit= 500,
    save_path="results/multiagent_results_4.csv",
    return_intermediates=True
)

df_out.head()

2025-08-25 10:32:34.508019: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756110754.525482  322703 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756110754.530325  322703 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1756110754.544237  322703 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1756110754.544248  322703 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1756110754.544250  322703 computation_placer.cc:177] computation placer alr

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore


📊 Evaluation
accuracy: 0.504
precision_true: 0.558
recall_true: 0.518
f1_true: 0.537
confusion_matrix: [[144 134]
 [114 108]]

Classification report:
               precision    recall  f1-score   support

        True       0.56      0.52      0.54       278
       False       0.45      0.49      0.47       222

    accuracy                           0.50       500
   macro avg       0.50      0.50      0.50       500
weighted avg       0.51      0.50      0.51       500

💾 Ergebnisse gespeichert unter: results/multiagent_results_4.csv


,label,statement,subjects,prediction,explanation,experts
0,True,Building a wall on the U.S . Mexico border wil...,immigration,False,Step 1: Identify the claim and subject The cla...,"[{'name': 'Politics&Elections SME', 'weight': ..."
1,False,Wisconsin is on pace to double the number of l...,jobs,False,Step 1: Identify the claim and subject The cla...,"[{'name': 'Politics&Elections SME', 'weight': ..."
2,False,Says John McCain has done nothing to help the ...,"military,veterans,voting-record",False,Step 1: Identify the claim and subject The cla...,"[{'name': 'Politics&Elections SME', 'weight': ..."
3,True,Suzanne Bonamici supports a plan that will cut...,"medicare,message-machine-2012,campaign-adverti...",True,The majority of experts agree that the claim i...,"[{'name': 'Politics&Elections SME', 'weight': ..."
4,False,When asked by a reporter whether hes at the ce...,"campaign-finance,legal-issues,campaign-adverti...",False,Here is the code to solve the problem: ```pyth...,"[{'name': 'Politics&Elections SME', 'weight': ..."


In [1]:
from multiagent_factcheck_4 import ExpertConfig, DecisionConfig, run_multiagent_on_csv

experts = [
    ExpertConfig(
        name="Politics&Elections SME",
        model_name="models/finetuned_with_subjects_json_style_politics_expert",
        weight=1.4,
        max_new_tokens=112,
        top_p=1.0,
        #device_map="cuda:0",
        prompt_template="""You are a subject-matter expert in U.S. politics, elections, legislation, and public records.

Task: Verify the claim with objective, publicly verifiable facts (official roll calls, bill texts, FEC/CRP, governor/congress records, reputable outlets). If evidence is insufficient/ambiguous, output "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Focus domains (if present): elections, campaign-finance, voting records, state/federal legislation, party platforms.

Input:
Claim: {statement}
{subject_block}
JSON:"""
    ),
    ExpertConfig(
        name="Economy&Taxes SME",
        model_name="models/finetuned_with_subjects_json_style_economy_expert",
        weight=1.3,
        max_new_tokens=112,
        top_p=1.0,
        #device_map="cuda:0",
        prompt_template="""You are a subject-matter expert in macroeconomics, labor stats, budgets, and taxation.

Task: Verify using BLS, BEA, CBO, JCT, Treasury/IRS, OMB, WTO/IMF/World Bank, and reputable reports. Quantify time ranges and definitions. If data is unclear or cherry-picked, output "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Focus domains: taxes, jobs, economy, trade, social security/medicare finances, government budget.

Input:
Claim: {statement}
{subject_block}
JSON:"""
    ),
    ExpertConfig(
        name="Health&Science SME",
        model_name="models/finetuned_with_subjects_json_style_health_expert",
        weight=1.3,
        max_new_tokens=112,
        top_p=1.0,
        #device_map="cuda:0",
        prompt_template="""You are a subject-matter expert for healthcare policy, public health, and scientific claims.

Task: Verify with CDC, FDA, NIH, CMS, WHO, Cochrane, PubMed-indexed studies, and official rulemaking. Distinguish correlation vs. causation and specify years/locations. If evidence is weak, output "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Focus domains: health-care, ACA/Medicare/Medicaid, vaccines, drugs/overdoses, abortion/stem cells, general science.

Input:
Claim: {statement}
{subject_block}
JSON:"""
    ),
]

decision = DecisionConfig(model_name="meta-llama/Llama-3.1-8B-Instruct")

df_out, metrics = run_multiagent_on_csv(
    csv_path="own_datasets/test_binary_labels.csv",
    experts=experts,
    decision=decision,
    statement_col="statement",
    subject_col="subjects",
    label_col="label",
    limit= 500,
    save_path="results/multiagent_results_finetuned_3.csv",
    return_intermediates=True
)

df_out.head()

2025-08-25 14:05:32.074872: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756123532.094923  325869 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756123532.099876  325869 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1756123532.114399  325869 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1756123532.114411  325869 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1756123532.114413  325869 computation_placer.cc:177] computation placer alr

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🔎 Fact-checking: 100%|██████████| 500/500 [2:47:55<00:00, 20.15s/it]  


📊 Evaluation
accuracy: 0.574
precision_true: 0.629
recall_true: 0.568
f1_true: 0.597
confusion_matrix: [[158 120]
 [ 93 129]]

Classification report:
               precision    recall  f1-score   support

        True       0.63      0.57      0.60       278
       False       0.52      0.58      0.55       222

    accuracy                           0.57       500
   macro avg       0.57      0.57      0.57       500
weighted avg       0.58      0.57      0.58       500

💾 Ergebnisse gespeichert unter: results/multiagent_results_finetuned_3.csv


,label,statement,subjects,prediction,explanation,experts
0,True,Building a wall on the U.S . Mexico border wil...,immigration,False,No explanation provided.,"[{'name': 'Politics&Elections SME', 'weight': ..."
1,False,Wisconsin is on pace to double the number of l...,jobs,True,The majority of experts with higher weights ag...,"[{'name': 'Politics&Elections SME', 'weight': ..."
2,False,Says John McCain has done nothing to help the ...,"military,veterans,voting-record",False,No explanation provided.,"[{'name': 'Politics&Elections SME', 'weight': ..."
3,True,Suzanne Bonamici supports a plan that will cut...,"medicare,message-machine-2012,campaign-adverti...",False,No explanation provided.,"[{'name': 'Politics&Elections SME', 'weight': ..."
4,False,When asked by a reporter whether hes at the ce...,"campaign-finance,legal-issues,campaign-adverti...",True,All three experts with high weights agree that...,"[{'name': 'Politics&Elections SME', 'weight': ..."


In [1]:
from multiagent_factcheck_5 import ExpertConfig, DecisionConfig, run_multiagent_on_csv

experts = [
    ExpertConfig(
        name="Politics&Elections SME",
        model_name="meta-llama/Llama-3.1-8B-Instruct",
        weight=1.4,
        max_new_tokens=112,
        top_p=1.0,
        prompt_template="""You are a subject-matter expert in U.S. politics, elections, legislation, and public records.

Task: Verify the claim with objective, publicly verifiable facts (official roll calls, bill texts, FEC/CRP, governor/congress records, reputable outlets). If evidence is insufficient/ambiguous, output "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Focus domains (if present): elections, campaign-finance, voting records, state/federal legislation, party platforms.

Input:
Claim: {statement}
{subject_block}
JSON:"""
    ),
    ExpertConfig(
        name="Economy&Taxes SME",
        model_name="meta-llama/Llama-3.1-8B-Instruct",
        weight=1.3,
        max_new_tokens=112,
        top_p=1.0,
        prompt_template="""You are a subject-matter expert in macroeconomics, labor stats, budgets, and taxation.

Task: Verify using BLS, BEA, CBO, JCT, Treasury/IRS, OMB, WTO/IMF/World Bank, and reputable reports. Quantify time ranges and definitions. If data is unclear or cherry-picked, output "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Focus domains: taxes, jobs, economy, trade, social security/medicare finances, government budget.

Input:
Claim: {statement}
{subject_block}
JSON:"""
    ),
    ExpertConfig(
        name="Health&Science SME",
        model_name="meta-llama/Llama-3.1-8B-Instruct",
        weight=1.3,
        max_new_tokens=112,
        top_p=1.0,
        prompt_template="""You are a subject-matter expert for healthcare policy, public health, and scientific claims.

Task: Verify with CDC, FDA, NIH, CMS, WHO, Cochrane, PubMed-indexed studies, and official rulemaking. Distinguish correlation vs. causation and specify years/locations. If evidence is weak, output "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Focus domains: health-care, ACA/Medicare/Medicaid, vaccines, drugs/overdoses, abortion/stem cells, general science.

Input:
Claim: {statement}
{subject_block}
JSON:"""
    ),
]

decision = DecisionConfig(
    model_name="meta-llama/Llama-3.1-8B-Instruct",
    prompt_template="""You are a senior fact-checking adjudicator.
You receive multiple expert analyses, each with a verdict, explanation, and weight.

Task:
- Evaluate the experts' reasoning quality, factual grounding, and agreement with publicly verifiable evidence.
- Consider the explicit weights assigned to each expert.
- Prioritize well-supported, domain-relevant arguments over unsupported or vague statements.
- If the majority view (weighted) is inconclusive or poorly justified, choose "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Input JSON (array of expert objects):
{experts_json}
JSON:"""
)


df_out, metrics = run_multiagent_on_csv(
    csv_path="own_datasets/test_binary_labels.csv",
    experts=experts,
    decision=decision,
    statement_col="statement",
    subject_col="subjects",
    label_col="label",
    limit= 500,
    save_path="results/multiagent_results_5.csv",
    return_intermediates=True
)

df_out.head()

2025-08-31 07:15:47.874632: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756617347.892068  329939 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756617347.896978  329939 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1756617347.911415  329939 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1756617347.911428  329939 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1756617347.911431  329939 computation_placer.cc:177] computation placer alr

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🔎 Fact-checking: 100%|██████████| 20/20 [08:19<00:00, 24.98s/it]


📊 Evaluation
accuracy: 0.300
precision_true: 0.273
recall_true: 0.333
f1_true: 0.300
confusion_matrix: [[3 6]
 [8 3]]

Classification report:
               precision    recall  f1-score   support

        True       0.27      0.33      0.30         9
       False       0.33      0.27      0.30        11

    accuracy                           0.30        20
   macro avg       0.30      0.30      0.30        20
weighted avg       0.31      0.30      0.30        20

💾 Ergebnisse gespeichert unter: results/multiagent_results_5.csv


,label,statement,subjects,prediction,explanation,experts,decision_raw_text,experts_raw_text
0,True,Building a wall on the U.S . Mexico border wil...,immigration,True,The majority of experts agree that the claim i...,"[{'name': 'Politics&Elections SME', 'weight': ...","{""verdict"": ""True"", ""explanation"": ""The major...","["" {\""claim\"": \""Building a wall on the U.S. M..."
1,False,Wisconsin is on pace to double the number of l...,jobs,True,The majority of experts agree that Wisconsin i...,"[{'name': 'Politics&Elections SME', 'weight': ...","{""verdict"": ""True"", ""explanation"": ""The major...","["" {\""claim\"": \""Wisconsin is on pace to doubl..."
2,False,Says John McCain has done nothing to help the ...,"military,veterans,voting-record",True,The majority of experts agree that John McCain...,"[{'name': 'Politics&Elections SME', 'weight': ...","{""verdict"": ""True"", ""explanation"": ""The major...","["" {\""claim\"": \""Says John McCain has done not..."
3,True,Suzanne Bonamici supports a plan that will cut...,"medicare,message-machine-2012,campaign-adverti...",False,The majority of experts agree that Suzanne Bon...,"[{'name': 'Politics&Elections SME', 'weight': ...","{""verdict"": ""True"", ""explanation"": ""The major...","["" {\""claim\"": \""Suzanne Bonamici supports a p..."
4,False,When asked by a reporter whether hes at the ce...,"campaign-finance,legal-issues,campaign-adverti...",True,The majority of experts agree that Gov. Scott ...,"[{'name': 'Politics&Elections SME', 'weight': ...","{""verdict"": ""True"", ""explanation"": ""The major...","["" {\""claim\"": \""When asked by a reporter whet..."


In [2]:
from multiagent_factcheck_5 import ExpertConfig, DecisionConfig, run_multiagent_on_csv

experts = [
    ExpertConfig(
        name="Politics&Elections SME",
        model_name="models/finetuned_with_subjects_json_style_politics_expert",
        weight=1.4,
        max_new_tokens=112,
        top_p=1.0,
        #device_map="cuda:0",
        prompt_template="""You are a subject-matter expert in U.S. politics, elections, legislation, and public records.

Task: Verify the claim with objective, publicly verifiable facts (official roll calls, bill texts, FEC/CRP, governor/congress records, reputable outlets). If evidence is insufficient/ambiguous, output "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Focus domains (if present): elections, campaign-finance, voting records, state/federal legislation, party platforms.

Input:
Claim: {statement}
{subject_block}
JSON:"""
    ),
    ExpertConfig(
        name="Economy&Taxes SME",
        model_name="models/finetuned_with_subjects_json_style_economy_expert",
        weight=1.3,
        max_new_tokens=112,
        top_p=1.0,
        #device_map="cuda:0",
        prompt_template="""You are a subject-matter expert in macroeconomics, labor stats, budgets, and taxation.

Task: Verify using BLS, BEA, CBO, JCT, Treasury/IRS, OMB, WTO/IMF/World Bank, and reputable reports. Quantify time ranges and definitions. If data is unclear or cherry-picked, output "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Focus domains: taxes, jobs, economy, trade, social security/medicare finances, government budget.

Input:
Claim: {statement}
{subject_block}
JSON:"""
    ),
    ExpertConfig(
        name="Health&Science SME",
        model_name="models/finetuned_with_subjects_json_style_health_expert",
        weight=1.3,
        max_new_tokens=112,
        top_p=1.0,
        #device_map="cuda:0",
        prompt_template="""You are a subject-matter expert for healthcare policy, public health, and scientific claims.

Task: Verify with CDC, FDA, NIH, CMS, WHO, Cochrane, PubMed-indexed studies, and official rulemaking. Distinguish correlation vs. causation and specify years/locations. If evidence is weak, output "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Focus domains: health-care, ACA/Medicare/Medicaid, vaccines, drugs/overdoses, abortion/stem cells, general science.

Input:
Claim: {statement}
{subject_block}
JSON:"""
    ),
]

decision = DecisionConfig(model_name="meta-llama/Llama-3.1-8B-Instruct")

df_out, metrics = run_multiagent_on_csv(
    csv_path="own_datasets/test_binary_labels.csv",
    experts=experts,
    decision=decision,
    statement_col="statement",
    subject_col="subjects",
    label_col="label",
    limit= 500,
    save_path="results/multiagent_results_finetuned_5.csv",
    return_intermediates=True
)

df_out.head()

🔎 Fact-checking:   0%|          | 0/20 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🔎 Fact-checking: 100%|██████████| 20/20 [07:10<00:00, 21.54s/it]


📊 Evaluation
accuracy: 0.500
precision_true: 0.429
recall_true: 0.333
f1_true: 0.375
confusion_matrix: [[3 6]
 [4 7]]

Classification report:
               precision    recall  f1-score   support

        True       0.43      0.33      0.38         9
       False       0.54      0.64      0.58        11

    accuracy                           0.50        20
   macro avg       0.48      0.48      0.48        20
weighted avg       0.49      0.50      0.49        20

💾 Ergebnisse gespeichert unter: results/multiagent_results_finetuned_5.csv


,label,statement,subjects,prediction,explanation,experts,decision_raw_text,experts_raw_text
0,True,Building a wall on the U.S . Mexico border wil...,immigration,False,"All three experts, with a combined weight of 4...","[{'name': 'Politics&Elections SME', 'weight': ...","BEGIN_DECISION_JSON\n{""verdict"": ""True"" or ""Fa...","[""{\""verdict\"": \""False\"", \""explanation\"": \""..."
1,False,Wisconsin is on pace to double the number of l...,jobs,False,The majority of experts with higher weights do...,"[{'name': 'Politics&Elections SME', 'weight': ...","BEGIN_DECISION_JSON\n{""verdict"": ""True"", ""expl...","[""{\""verdict\"": \""True\"", \""explanation\"": \""I..."
2,False,Says John McCain has done nothing to help the ...,"military,veterans,voting-record",False,The lack of evidence supporting the claim in m...,"[{'name': 'Politics&Elections SME', 'weight': ...","BEGIN_DECISION_JSON\n{""verdict"": ""True"", ""expl...","[""{\""verdict\"": \""False\"", \""explanation\"": \""..."
3,True,Suzanne Bonamici supports a plan that will cut...,"medicare,message-machine-2012,campaign-adverti...",False,The lack of evidence supporting the claim in m...,"[{'name': 'Politics&Elections SME', 'weight': ...","BEGIN_DECISION_JSON\n{""verdict"": ""True"", ""expl...","[""{\""verdict\"": \""False\"", \""explanation\"": \""..."
4,False,When asked by a reporter whether hes at the ce...,"campaign-finance,legal-issues,campaign-adverti...",False,"The lack of evidence from most experts, partic...","[{'name': 'Politics&Elections SME', 'weight': ...","BEGIN_DECISION_JSON\n{""verdict"": ""True"", ""expl...","[""{\""verdict\"": \""True\"", \""explanation\"": \""I..."


# anderes als domänen experten

In [1]:
from multiagent_factcheck_4 import ExpertConfig, DecisionConfig, run_multiagent_on_csv

experts = [
    ExpertConfig(
    name="Rhetoric&Style SME",
    model_name="meta-llama/Llama-3.1-8B-Instruct",
    weight=1.0,
    max_new_tokens=112,
    top_p=1.0,
    prompt_template="""You are a subject-matter expert for rhetoric, persuasive language, and discourse analysis.

Task: Analyze the claim's language for rhetorical devices (absolutism, hyperbole, sarcasm, emotionally loaded words, false balance).
- If the claim relies mainly on rhetoric without verifiable evidence, output "False".
- If rhetoric is minimal and factual basis is present, do not penalize.

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Input:
Claim: {statement}
{subject_block}
JSON:"""
),
    
ExpertConfig(
    name="Logic&Reasoning SME",
    model_name="meta-llama/Llama-3.1-8B-Instruct",
    weight=1.0,
    max_new_tokens=112,
    top_p=1.0,
    prompt_template="""You are a subject-matter expert in logic, reasoning fallacies, and argumentation theory.

Task: Check if the claim is logically consistent.
- Detect contradictions, overgeneralizations from single events, tautologies, circular reasoning, false cause/effect.
- If the claim is logically unsound or cannot be generalized, output "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Input:
Claim: {statement}
{subject_block}
JSON:"""
)
,
   ExpertConfig(
    name="KnowledgeType SME",
    model_name="meta-llama/Llama-3.1-8B-Instruct",
    weight=1.0,
    max_new_tokens=112,
    top_p=1.0,
    prompt_template="""You are a subject-matter expert in epistemology: distinguishing factual claims from opinions or unverifiable assertions.

Task: Assess if the claim is:
- A checkable fact with objective evidence,
- A subjective opinion,
- Ambiguous or context-dependent.

If the claim is opinion, unverifiable, or ambiguous, output "False". If it's clearly checkable as fact, output "True".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Input:
Claim: {statement}
{subject_block}
JSON:"""
)
,
    ExpertConfig(
    name="Verifiability SME",
    model_name="meta-llama/Llama-3.1-8B-Instruct",
    weight=1.0,
    max_new_tokens=112,
    top_p=1.0,
    prompt_template="""You are a subject-matter expert for claim verifiability.

Task: Judge whether the claim is formulated in a way that can be independently verified by evidence.
- If the claim is vague, lacks measurable criteria, or is framed as a non-falsifiable opinion, output "False".
- If the claim is specific, factual, and checkable, output "True".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Input:
Claim: {statement}
{subject_block}
JSON:"""
)

]

decision = DecisionConfig(
    model_name="meta-llama/Llama-3.1-8B-Instruct",
    prompt_template="""You are a senior fact-checking adjudicator.
You receive multiple expert analyses, each with a verdict, explanation, and weight.

Task:
- Evaluate the experts' reasoning quality, factual grounding, and agreement with publicly verifiable evidence.
- Consider the explicit weights assigned to each expert.
- Prioritize well-supported, domain-relevant arguments over unsupported or vague statements.
- If the majority view (weighted) is inconclusive or poorly justified, choose "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Input JSON (array of expert objects):
{experts_json}
JSON:"""
)


df_out, metrics = run_multiagent_on_csv(
    csv_path="own_datasets/test_binary_labels.csv",
    experts=experts,
    decision=decision,
    statement_col="statement",
    subject_col="subjects",
    label_col="label",
    limit= 500,
    save_path="results/multiagent_results_other_agents.csv",
    return_intermediates=True
)

df_out.head()

2025-08-31 11:19:39.353465: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756631979.370670  333791 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756631979.375461  333791 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1756631979.388899  333791 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1756631979.388910  333791 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1756631979.388912  333791 computation_placer.cc:177] computation placer alr

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🔎 Fact-checking: 100%|██████████| 500/500 [3:59:44<00:00, 28.77s/it]  


📊 Evaluation
accuracy: 0.428
precision_true: 0.470
recall_true: 0.227
f1_true: 0.306
confusion_matrix: [[ 63 215]
 [ 71 151]]

Classification report:
               precision    recall  f1-score   support

        True       0.47      0.23      0.31       278
       False       0.41      0.68      0.51       222

    accuracy                           0.43       500
   macro avg       0.44      0.45      0.41       500
weighted avg       0.44      0.43      0.40       500

💾 Ergebnisse gespeichert unter: results/multiagent_results_other_agents.csv


,label,statement,subjects,prediction,explanation,experts
0,True,Building a wall on the U.S . Mexico border wil...,immigration,False,Here is the analysis of the claim: The claim u...,"[{'name': 'Rhetoric&Style SME', 'weight': 1.0,..."
1,False,Wisconsin is on pace to double the number of l...,jobs,False,"{""experts"": [ , , { ""name"": ""KnowledgeType SME...","[{'name': 'Rhetoric&Style SME', 'weight': 1.0,..."
2,False,Says John McCain has done nothing to help the ...,"military,veterans,voting-record",True,The majority of experts agree that the claim i...,"[{'name': 'Rhetoric&Style SME', 'weight': 1.0,..."
3,True,Suzanne Bonamici supports a plan that will cut...,"medicare,message-machine-2012,campaign-adverti...",True,The claim is a statement about Suzanne Bonamic...,"[{'name': 'Rhetoric&Style SME', 'weight': 1.0,..."
4,False,When asked by a reporter whether hes at the ce...,"campaign-finance,legal-issues,campaign-adverti...",False,"{""experts"": [ , , { ""name"": ""KnowledgeType SME...","[{'name': 'Rhetoric&Style SME', 'weight': 1.0,..."


In [ ]:
from multiagent_factcheck_5 import ExpertConfig, DecisionConfig, run_multiagent_on_csv

experts = [
    ExpertConfig(
        name="Politics&Elections SME",
        model_name="meta-llama/Llama-3.1-8B-Instruct",
        weight=1.4,
        max_new_tokens=112,
        top_p=1.0,
        prompt_template="""You are a subject-matter expert in U.S. politics, elections, legislation, and public records.

Task: Verify the claim with objective, publicly verifiable facts (official roll calls, bill texts, FEC/CRP, governor/congress records, reputable outlets). If evidence is insufficient/ambiguous, output "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Focus domains (if present): elections, campaign-finance, voting records, state/federal legislation, party platforms.

Input:
Claim: {statement}
{subject_block}
JSON:"""
    ),
    ExpertConfig(
        name="Economy&Taxes SME",
        model_name="meta-llama/Llama-3.1-8B-Instruct",
        weight=1.3,
        max_new_tokens=112,
        top_p=1.0,
        prompt_template="""You are a subject-matter expert in macroeconomics, labor stats, budgets, and taxation.

Task: Verify using BLS, BEA, CBO, JCT, Treasury/IRS, OMB, WTO/IMF/World Bank, and reputable reports. Quantify time ranges and definitions. If data is unclear or cherry-picked, output "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Focus domains: taxes, jobs, economy, trade, social security/medicare finances, government budget.

Input:
Claim: {statement}
{subject_block}
JSON:"""
    ),
    ExpertConfig(
        name="Health&Science SME",
        model_name="meta-llama/Llama-3.1-8B-Instruct",
        weight=1.3,
        max_new_tokens=112,
        top_p=1.0,
        prompt_template="""You are a subject-matter expert for healthcare policy, public health, and scientific claims.

Task: Verify with CDC, FDA, NIH, CMS, WHO, Cochrane, PubMed-indexed studies, and official rulemaking. Distinguish correlation vs. causation and specify years/locations. If evidence is weak, output "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Focus domains: health-care, ACA/Medicare/Medicaid, vaccines, drugs/overdoses, abortion/stem cells, general science.

Input:
Claim: {statement}
{subject_block}
JSON:"""
    ),
        ExpertConfig(
    name="Rhetoric&Style SME",
    model_name="meta-llama/Llama-3.1-8B-Instruct",
    weight=1.0,
    max_new_tokens=112,
    top_p=1.0,
    prompt_template="""You are a subject-matter expert for rhetoric, persuasive language, and discourse analysis.

Task: Analyze the claim's language for rhetorical devices (absolutism, hyperbole, sarcasm, emotionally loaded words, false balance).
- If the claim relies mainly on rhetoric without verifiable evidence, output "False".
- If rhetoric is minimal and factual basis is present, do not penalize.

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Input:
Claim: {statement}
{subject_block}
JSON:"""
)
]

decision = DecisionConfig(
    model_name="meta-llama/Llama-3.1-8B-Instruct",
    prompt_template="""You are a senior fact-checking adjudicator.
You receive multiple expert analyses, each with a verdict, explanation, and weight.

Task:
- Evaluate the experts' reasoning quality, factual grounding, and agreement with publicly verifiable evidence.
- Consider the explicit weights assigned to each expert.
- Prioritize well-supported, domain-relevant arguments over unsupported or vague statements.
- If the majority view (weighted) is inconclusive or poorly justified, choose "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Input JSON (array of expert objects):
{experts_json}
JSON:"""
)


df_out, metrics = run_multiagent_on_csv(
    csv_path="own_datasets/test_binary_labels.csv",
    experts=experts,
    decision=decision,
    statement_col="statement",
    subject_col="subjects",
    label_col="label",
    limit= 500,
    save_path="results/multiagent_results__andere_kombi_1.csv",
    return_intermediates=True
)

df_out.head()

2025-08-31 15:27:32.606035: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756646852.624683  337012 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756646852.629701  337012 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1756646852.644269  337012 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1756646852.644284  337012 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1756646852.644286  337012 computation_placer.cc:177] computation placer alr

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🔎 Fact-checking:   1%|          | 4/500 [02:12<4:17:52, 31.19s/it]

In [2]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

# Pfad zur Ergebnis-CSV
csv_path = "results/multiagent_results__andere_kombi_1.csv"

# CSV einlesen
df = pd.read_csv(csv_path)

# Ground Truth und Vorhersagen als Strings normalisieren
y_true = df["label"].astype(str)
y_pred = df["prediction"].astype(str)

# Metriken berechnen
metrics = {
    "accuracy": accuracy_score(y_true, y_pred),
    "precision_true": precision_score(y_true, y_pred, pos_label="True"),
    "recall_true": recall_score(y_true, y_pred, pos_label="True"),
    "f1_true": f1_score(y_true, y_pred, pos_label="True"),
    "report": classification_report(y_true, y_pred, labels=["True","False"], target_names=["True","False"]),
    "confusion_matrix": confusion_matrix(y_true, y_pred, labels=["True", "False"])
}

# Ergebnisse ausgeben
print("\n📊 Evaluation")
for k, v in metrics.items():
    if k != "report":
        print(f"{k}: {v:.3f}" if isinstance(v, float) else f"{k}: {v}")
print("\nClassification report:\n", metrics["report"])



📊 Evaluation
accuracy: 0.476
precision_true: 0.573
recall_true: 0.227
f1_true: 0.325
confusion_matrix: [[ 63 215]
 [ 47 175]]

Classification report:
               precision    recall  f1-score   support

        True       0.57      0.23      0.32       278
       False       0.45      0.79      0.57       222

    accuracy                           0.48       500
   macro avg       0.51      0.51      0.45       500
weighted avg       0.52      0.48      0.43       500



In [1]:
from multiagent_factcheck_5 import ExpertConfig, DecisionConfig, run_multiagent_on_csv

experts = [
    ExpertConfig(
        name="Politics&Elections SME",
        model_name="meta-llama/Llama-3.1-8B-Instruct",
        weight=1.4,
        max_new_tokens=112,
        top_p=1.0,
        prompt_template="""You are a subject-matter expert in U.S. politics, elections, legislation, and public records.

Task: Verify the claim with objective, publicly verifiable facts (official roll calls, bill texts, FEC/CRP, governor/congress records, reputable outlets). If evidence is insufficient/ambiguous, output "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Focus domains (if present): elections, campaign-finance, voting records, state/federal legislation, party platforms.

Input:
Claim: {statement}
{subject_block}
JSON:"""
    ),
    ExpertConfig(
        name="Economy&Taxes SME",
        model_name="meta-llama/Llama-3.1-8B-Instruct",
        weight=1.3,
        max_new_tokens=112,
        top_p=1.0,
        prompt_template="""You are a subject-matter expert in macroeconomics, labor stats, budgets, and taxation.

Task: Verify using BLS, BEA, CBO, JCT, Treasury/IRS, OMB, WTO/IMF/World Bank, and reputable reports. Quantify time ranges and definitions. If data is unclear or cherry-picked, output "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Focus domains: taxes, jobs, economy, trade, social security/medicare finances, government budget.

Input:
Claim: {statement}
{subject_block}
JSON:"""
    ),
    ExpertConfig(
        name="Health&Science SME",
        model_name="meta-llama/Llama-3.1-8B-Instruct",
        weight=1.3,
        max_new_tokens=112,
        top_p=1.0,
        prompt_template="""You are a subject-matter expert for healthcare policy, public health, and scientific claims.

Task: Verify with CDC, FDA, NIH, CMS, WHO, Cochrane, PubMed-indexed studies, and official rulemaking. Distinguish correlation vs. causation and specify years/locations. If evidence is weak, output "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Focus domains: health-care, ACA/Medicare/Medicaid, vaccines, drugs/overdoses, abortion/stem cells, general science.

Input:
Claim: {statement}
{subject_block}
JSON:"""
    ),
ExpertConfig(
    name="Logic&Reasoning SME",
    model_name="meta-llama/Llama-3.1-8B-Instruct",
    weight=1.0,
    max_new_tokens=112,
    top_p=1.0,
    prompt_template="""You are a subject-matter expert in logic, reasoning fallacies, and argumentation theory.

Task: Check if the claim is logically consistent.
- Detect contradictions, overgeneralizations from single events, tautologies, circular reasoning, false cause/effect.
- If the claim is logically unsound or cannot be generalized, output "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Input:
Claim: {statement}
{subject_block}
JSON:"""
)
]

decision = DecisionConfig(
    model_name="meta-llama/Llama-3.1-8B-Instruct",
    prompt_template="""You are a senior fact-checking adjudicator.
You receive multiple expert analyses, each with a verdict, explanation, and weight.

Task:
- Evaluate the experts' reasoning quality, factual grounding, and agreement with publicly verifiable evidence.
- Consider the explicit weights assigned to each expert.
- Prioritize well-supported, domain-relevant arguments over unsupported or vague statements.
- If the majority view (weighted) is inconclusive or poorly justified, choose "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Input JSON (array of expert objects):
{experts_json}
JSON:"""
)


df_out, metrics = run_multiagent_on_csv(
    csv_path="own_datasets/test_binary_labels.csv",
    experts=experts,
    decision=decision,
    statement_col="statement",
    subject_col="subjects",
    label_col="label",
    limit= 500,
    save_path="results/multiagent_results__andere_kombi_2.csv",
    return_intermediates=True
)

df_out.head()

2025-09-01 08:54:27.707884: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756709667.725081  339873 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756709667.729847  339873 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1756709667.743202  339873 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1756709667.743215  339873 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1756709667.743217  339873 computation_placer.cc:177] computation placer alr

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🔎 Fact-checking: 100%|██████████| 500/500 [4:00:02<00:00, 28.80s/it]  


📊 Evaluation
accuracy: 0.478
precision_true: 0.577
recall_true: 0.230
f1_true: 0.329
confusion_matrix: [[ 64 214]
 [ 47 175]]

Classification report:
               precision    recall  f1-score   support

        True       0.58      0.23      0.33       278
       False       0.45      0.79      0.57       222

    accuracy                           0.48       500
   macro avg       0.51      0.51      0.45       500
weighted avg       0.52      0.48      0.44       500

💾 Ergebnisse gespeichert unter: results/multiagent_results__andere_kombi_2.csv


,label,statement,subjects,prediction,explanation,experts,decision_raw_text,experts_raw_text
0,True,Building a wall on the U.S . Mexico border wil...,immigration,False,"{""experts"": [ , { ""name"": ""Economy&Taxes SME"",...","[{'name': 'Politics&Elections SME', 'weight': ...","{""experts"": [\n {\n ""name"": ""Politics&Ele...","["" {\""claim\"": \""Building a wall on the U.S. M..."
1,False,Wisconsin is on pace to double the number of l...,jobs,False,"[ , { ""name"": ""Economy&Taxes SME"", ""weight"": 1...","[{'name': 'Politics&Elections SME', 'weight': ...","\n[\n {\n ""name"": ""Politics&Elections SME...","["" {\""claim\"": \""Wisconsin is on pace to doubl..."
2,False,Says John McCain has done nothing to help the ...,"military,veterans,voting-record",True,John McCain has a long history of advocating f...,"[{'name': 'Politics&Elections SME', 'weight': ...","{""verdict"": ""True"", ""explanation"": ""John McCa...","["" {\""claim\"": \""Says John McCain has done not..."
3,True,Suzanne Bonamici supports a plan that will cut...,"medicare,message-machine-2012,campaign-adverti...",False,The majority of experts agree that Suzanne Bon...,"[{'name': 'Politics&Elections SME', 'weight': ...","{""verdict"": ""True"", ""explanation"": ""Suzanne B...","["" {\""claim\"": \""Suzanne Bonamici supports a p..."
4,False,When asked by a reporter whether hes at the ce...,"campaign-finance,legal-issues,campaign-adverti...",False,"{""experts"": [ { ""name"": ""Politics&Elections SM...","[{'name': 'Politics&Elections SME', 'weight': ...","{""experts"": [\n {\n ""name"": ""Politics&Ele...","["" {\""claim\"": \""When asked by a reporter whet..."


In [ ]:
from multiagent_factcheck_5 import ExpertConfig, DecisionConfig, run_multiagent_on_csv

experts = [
    ExpertConfig(
        name="Politics&Elections SME",
        model_name="meta-llama/Llama-3.1-8B-Instruct",
        weight=1.4,
        max_new_tokens=112,
        top_p=1.0,
        prompt_template="""You are a subject-matter expert in U.S. politics, elections, legislation, and public records.

Task: Verify the claim with objective, publicly verifiable facts (official roll calls, bill texts, FEC/CRP, governor/congress records, reputable outlets). If evidence is insufficient/ambiguous, output "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Focus domains (if present): elections, campaign-finance, voting records, state/federal legislation, party platforms.

Input:
Claim: {statement}
{subject_block}
JSON:"""
    ),
    ExpertConfig(
        name="Economy&Taxes SME",
        model_name="meta-llama/Llama-3.1-8B-Instruct",
        weight=1.3,
        max_new_tokens=112,
        top_p=1.0,
        prompt_template="""You are a subject-matter expert in macroeconomics, labor stats, budgets, and taxation.

Task: Verify using BLS, BEA, CBO, JCT, Treasury/IRS, OMB, WTO/IMF/World Bank, and reputable reports. Quantify time ranges and definitions. If data is unclear or cherry-picked, output "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Focus domains: taxes, jobs, economy, trade, social security/medicare finances, government budget.

Input:
Claim: {statement}
{subject_block}
JSON:"""
    ),
    ExpertConfig(
        name="Health&Science SME",
        model_name="meta-llama/Llama-3.1-8B-Instruct",
        weight=1.3,
        max_new_tokens=112,
        top_p=1.0,
        prompt_template="""You are a subject-matter expert for healthcare policy, public health, and scientific claims.

Task: Verify with CDC, FDA, NIH, CMS, WHO, Cochrane, PubMed-indexed studies, and official rulemaking. Distinguish correlation vs. causation and specify years/locations. If evidence is weak, output "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Focus domains: health-care, ACA/Medicare/Medicaid, vaccines, drugs/overdoses, abortion/stem cells, general science.

Input:
Claim: {statement}
{subject_block}
JSON:"""
    ),
   ExpertConfig(
    name="KnowledgeType SME",
    model_name="meta-llama/Llama-3.1-8B-Instruct",
    weight=1.0,
    max_new_tokens=112,
    top_p=1.0,
    prompt_template="""You are a subject-matter expert in epistemology: distinguishing factual claims from opinions or unverifiable assertions.

Task: Assess if the claim is:
- A checkable fact with objective evidence,
- A subjective opinion,
- Ambiguous or context-dependent.

If the claim is opinion, unverifiable, or ambiguous, output "False". If it's clearly checkable as fact, output "True".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Input:
Claim: {statement}
{subject_block}
JSON:"""
)
]

decision = DecisionConfig(
    model_name="meta-llama/Llama-3.1-8B-Instruct",
    prompt_template="""You are a senior fact-checking adjudicator.
You receive multiple expert analyses, each with a verdict, explanation, and weight.

Task:
- Evaluate the experts' reasoning quality, factual grounding, and agreement with publicly verifiable evidence.
- Consider the explicit weights assigned to each expert.
- Prioritize well-supported, domain-relevant arguments over unsupported or vague statements.
- If the majority view (weighted) is inconclusive or poorly justified, choose "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Input JSON (array of expert objects):
{experts_json}
JSON:"""
)


df_out, metrics = run_multiagent_on_csv(
    csv_path="own_datasets/test_binary_labels.csv",
    experts=experts,
    decision=decision,
    statement_col="statement",
    subject_col="subjects",
    label_col="label",
    limit= 500,
    save_path="results/multiagent_results__andere_kombi_3.csv",
    return_intermediates=True
)

df_out.head()

2025-09-01 16:45:42.826057: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756737942.843358  345974 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756737942.848163  345974 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1756737942.861566  345974 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1756737942.861578  345974 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1756737942.861580  345974 computation_placer.cc:177] computation placer alr

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🔎 Fact-checking:   1%|          | 4/500 [02:12<4:16:46, 31.06s/it]

In [2]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

# Pfad zur Ergebnis-CSV
csv_path = "results/multiagent_results__andere_kombi_3.csv"

# CSV einlesen
df = pd.read_csv(csv_path)

# Ground Truth und Vorhersagen als Strings normalisieren
y_true = df["label"].astype(str)
y_pred = df["prediction"].astype(str)

# Metriken berechnen
metrics = {
    "accuracy": accuracy_score(y_true, y_pred),
    "precision_true": precision_score(y_true, y_pred, pos_label="True"),
    "recall_true": recall_score(y_true, y_pred, pos_label="True"),
    "f1_true": f1_score(y_true, y_pred, pos_label="True"),
    "report": classification_report(y_true, y_pred, labels=["True","False"], target_names=["True","False"]),
    "confusion_matrix": confusion_matrix(y_true, y_pred, labels=["True", "False"])
}

# Ergebnisse ausgeben
print("\n📊 Evaluation")
for k, v in metrics.items():
    if k != "report":
        print(f"{k}: {v:.3f}" if isinstance(v, float) else f"{k}: {v}")
print("\nClassification report:\n", metrics["report"])



📊 Evaluation
accuracy: 0.446
precision_true: 0.504
recall_true: 0.227
f1_true: 0.313
confusion_matrix: [[ 63 215]
 [ 62 160]]

Classification report:
               precision    recall  f1-score   support

        True       0.50      0.23      0.31       278
       False       0.43      0.72      0.54       222

    accuracy                           0.45       500
   macro avg       0.47      0.47      0.42       500
weighted avg       0.47      0.45      0.41       500



In [1]:
from multiagent_factcheck_5 import ExpertConfig, DecisionConfig, run_multiagent_on_csv

experts = [
    ExpertConfig(
        name="Politics&Elections SME",
        model_name="meta-llama/Llama-3.1-8B-Instruct",
        weight=1.4,
        max_new_tokens=112,
        top_p=1.0,
        prompt_template="""You are a subject-matter expert in U.S. politics, elections, legislation, and public records.

Task: Verify the claim with objective, publicly verifiable facts (official roll calls, bill texts, FEC/CRP, governor/congress records, reputable outlets). If evidence is insufficient/ambiguous, output "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Focus domains (if present): elections, campaign-finance, voting records, state/federal legislation, party platforms.

Input:
Claim: {statement}
{subject_block}
JSON:"""
    ),
    ExpertConfig(
        name="Economy&Taxes SME",
        model_name="meta-llama/Llama-3.1-8B-Instruct",
        weight=1.3,
        max_new_tokens=112,
        top_p=1.0,
        prompt_template="""You are a subject-matter expert in macroeconomics, labor stats, budgets, and taxation.

Task: Verify using BLS, BEA, CBO, JCT, Treasury/IRS, OMB, WTO/IMF/World Bank, and reputable reports. Quantify time ranges and definitions. If data is unclear or cherry-picked, output "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Focus domains: taxes, jobs, economy, trade, social security/medicare finances, government budget.

Input:
Claim: {statement}
{subject_block}
JSON:"""
    ),
    ExpertConfig(
        name="Health&Science SME",
        model_name="meta-llama/Llama-3.1-8B-Instruct",
        weight=1.3,
        max_new_tokens=112,
        top_p=1.0,
        prompt_template="""You are a subject-matter expert for healthcare policy, public health, and scientific claims.

Task: Verify with CDC, FDA, NIH, CMS, WHO, Cochrane, PubMed-indexed studies, and official rulemaking. Distinguish correlation vs. causation and specify years/locations. If evidence is weak, output "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Focus domains: health-care, ACA/Medicare/Medicaid, vaccines, drugs/overdoses, abortion/stem cells, general science.

Input:
Claim: {statement}
{subject_block}
JSON:"""
    ),
    ExpertConfig(
    name="Verifiability SME",
    model_name="meta-llama/Llama-3.1-8B-Instruct",
    weight=1.0,
    max_new_tokens=112,
    top_p=1.0,
    prompt_template="""You are a subject-matter expert for claim verifiability.

Task: Judge whether the claim is formulated in a way that can be independently verified by evidence.
- If the claim is vague, lacks measurable criteria, or is framed as a non-falsifiable opinion, output "False".
- If the claim is specific, factual, and checkable, output "True".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Input:
Claim: {statement}
{subject_block}
JSON:"""
)
]

decision = DecisionConfig(
    model_name="meta-llama/Llama-3.1-8B-Instruct",
    prompt_template="""You are a senior fact-checking adjudicator.
You receive multiple expert analyses, each with a verdict, explanation, and weight.

Task:
- Evaluate the experts' reasoning quality, factual grounding, and agreement with publicly verifiable evidence.
- Consider the explicit weights assigned to each expert.
- Prioritize well-supported, domain-relevant arguments over unsupported or vague statements.
- If the majority view (weighted) is inconclusive or poorly justified, choose "False".

Return ONLY this JSON:
{{"verdict":"True|False","explanation":"2-4 explaining sentences"}}

Input JSON (array of expert objects):
{experts_json}
JSON:"""
)


df_out, metrics = run_multiagent_on_csv(
    csv_path="own_datasets/test_binary_labels.csv",
    experts=experts,
    decision=decision,
    statement_col="statement",
    subject_col="subjects",
    label_col="label",
    limit= 500,
    save_path="results/multiagent_results__andere_kombi_4.csv",
    return_intermediates=True
)

df_out.head()

2025-09-02 09:31:57.193221: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756798317.210533  348191 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756798317.215337  348191 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1756798317.228662  348191 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1756798317.228674  348191 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1756798317.228676  348191 computation_placer.cc:177] computation placer alr

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🔎 Fact-checking: 100%|██████████| 500/500 [4:01:05<00:00, 28.93s/it]  


📊 Evaluation
accuracy: 0.464
precision_true: 0.543
recall_true: 0.227
f1_true: 0.320
confusion_matrix: [[ 63 215]
 [ 53 169]]

Classification report:
               precision    recall  f1-score   support

        True       0.54      0.23      0.32       278
       False       0.44      0.76      0.56       222

    accuracy                           0.46       500
   macro avg       0.49      0.49      0.44       500
weighted avg       0.50      0.46      0.43       500

💾 Ergebnisse gespeichert unter: results/multiagent_results__andere_kombi_4.csv


,label,statement,subjects,prediction,explanation,experts,decision_raw_text,experts_raw_text
0,True,Building a wall on the U.S . Mexico border wil...,immigration,False,"{""experts"": [ , { ""name"": ""Economy&Taxes SME"",...","[{'name': 'Politics&Elections SME', 'weight': ...","{""experts"": [\n {\n ""name"": ""Politics&Ele...","["" {\""claim\"": \""Building a wall on the U.S. M..."
1,False,Wisconsin is on pace to double the number of l...,jobs,False,"{""experts"": [ { ""name"": ""Politics&Elections SM...","[{'name': 'Politics&Elections SME', 'weight': ...","{""experts"": [\n {\n ""name"": ""Politics&Ele...","["" {\""claim\"": \""Wisconsin is on pace to doubl..."
2,False,Says John McCain has done nothing to help the ...,"military,veterans,voting-record",False,"{""experts"": [ { ""name"": ""Politics&Elections SM...","[{'name': 'Politics&Elections SME', 'weight': ...","{""experts"": [\n {\n ""name"": ""Politics&Ele...","["" {\""claim\"": \""Says John McCain has done not..."
3,True,Suzanne Bonamici supports a plan that will cut...,"medicare,message-machine-2012,campaign-adverti...",False,"[ , , { ""name"": ""Health&Science SME"", ""weight""...","[{'name': 'Politics&Elections SME', 'weight': ...","\n[\n {\n ""name"": ""Politics&Elections SME...","["" {\""claim\"": \""Suzanne Bonamici supports a p..."
4,False,When asked by a reporter whether hes at the ce...,"campaign-finance,legal-issues,campaign-adverti...",True,The majority of experts agree that the claim i...,"[{'name': 'Politics&Elections SME', 'weight': ...","{""verdict"": ""True"", ""explanation"": ""The major...","["" {\""claim\"": \""When asked by a reporter whet..."
